# 24A — Competition-only input preflight (CPU)

**Purpose:** inventory the attached competition data, label tables and any prebuilt MRI cache before spending GPU time. This notebook does **not** train, infer, or submit. It does not certify that an attached dataset's weights were trained without external MRI.

Attach the RSNA Knee competition data. Optionally attach a dataset containing your proposed **96-slice, 384×384** competition-only cache and report-derived training labels. Use **Accelerator: None; Internet: Off**. Save & Run All, then send `preflight_24a.json` from Output. A `NEEDS_INPUTS` status is an expected, successful audit result—not a reason to run GPU.


In [ ]:
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import json, os, shutil, time
import numpy as np
import pandas as pd

START = time.monotonic()
INPUT = Path('/kaggle/input')
OUT = Path('/kaggle/working/preflight_24a.json')
EXPECTED = {'studies_weak': 4349, 'studies_gold': 58, 'slices': 96, 'height': 384, 'width': 384}
report = {'experiment': '24A_competition_only_input_preflight', 'time_utc': datetime.now(timezone.utc).isoformat(), 'expected': EXPECTED, 'checks': {}, 'notes': []}
assert INPUT.is_dir(), '/kaggle/input is unavailable'
print('CPU preflight; CUDA is not used')


In [ ]:
# Discover only likely CSV locations; never walk the large DICOM train_series tree.
roots = [INPUT / 'competitions' / 'rsna-knee-abnormality-detection', INPUT / 'rsna-knee-abnormality-detection']
comp = next((p for p in roots if (p / 'train.csv').is_file()), None)
if comp is None:
    for top in INPUT.iterdir():
        if top.is_dir() and top.name not in ('competitions', 'datasets', 'notebooks') and (top / 'train.csv').is_file():
            comp = top; break
report['checks']['competition_root'] = str(comp) if comp else None
if comp:
    train = pd.read_csv(comp / 'train.csv')
    uid = 'StudyInstanceUID'
    report['checks']['train_csv'] = {'rows': int(len(train)), 'unique_studies': int(train[uid].astype(str).nunique()) if uid in train else None, 'columns': list(train.columns), 'non_null_per_column': {str(c): int(train[c].notna().sum()) for c in train.columns}}
    for name in ('train_series.csv', 'test.csv', 'sample_submission.csv'):
        p = comp / name
        if p.is_file():
            df = pd.read_csv(p, usecols=[uid]) if name != 'sample_submission.csv' else pd.read_csv(p, nrows=0)
            report['checks'][name] = {'rows': int(len(df)) if name != 'sample_submission.csv' else None, 'columns': list(df.columns)}
    report['checks']['competition_has_train_series_dir'] = (comp / 'train_series').is_dir()
else:
    report['notes'].append('Competition data not attached; attach RSNA Knee Abnormality Detection.')
print(json.dumps(report['checks'], indent=2)[:6000])


In [ ]:
# Bounded search across non-competition inputs. Do not recursively traverse DICOMs.
mounts = []
for family in ('datasets', 'notebooks'):
    base = INPUT / family
    if not base.is_dir(): continue
    for owner in base.iterdir():
        if not owner.is_dir(): continue
        for mount in owner.iterdir():
            if mount.is_dir(): mounts.append(mount)
for top in INPUT.iterdir():
    if top.is_dir() and top.name not in ('competitions', 'datasets', 'notebooks') and top != comp:
        mounts.append(top)
mounts = sorted(set(mounts), key=str)
report['checks']['noncompetition_mount_count'] = len(mounts)
cache_hits, table_hits = [], []
for mount in mounts:
    dirs = [mount]
    for level in range(3):
        new = []
        for d in dirs:
            if not d.is_dir(): continue
            try: entries = list(d.iterdir())
            except OSError: continue
            for p in entries:
                if p.is_file() and p.suffix.lower() in ('.csv', '.parquet', '.json') and any(k in p.name.lower() for k in ('label', 'report', 'manifest', 'fold', 'cache')):
                    table_hits.append({'path': str(p), 'bytes': p.stat().st_size})
                if p.is_dir() and p.name.lower() in ('study_cache', 'cache96', 'dense96', 'corpus96'):
                    cache_hits.append(p)
                elif p.is_dir() and level < 2 and p.name.lower() not in ('train_series', 'test_series', 'train_images', 'test_images'):
                    new.append(p)
        dirs = new
report['checks']['candidate_label_tables'] = table_hits[:100]
report['checks']['candidate_cache_dirs'] = []
for d in cache_hits:
    try:
        files = [p for p in d.iterdir() if p.is_file()]
        npzs = [p for p in files if p.suffix.lower() == '.npz']
        sample = []
        for p in npzs[:2]:
            try:
                with np.load(p, allow_pickle=False) as z:
                    sample.append({'file': p.name, 'arrays': {k: {'shape': list(z[k].shape), 'dtype': str(z[k].dtype)} for k in z.files if z[k].ndim >= 1}})
            except Exception as e:
                sample.append({'file': p.name, 'error': type(e).__name__ + ': ' + str(e)[:120]})
        report['checks']['candidate_cache_dirs'].append({'path': str(d), 'npz_count': len(npzs), 'total_npz_gib': round(sum(p.stat().st_size for p in npzs) / 2**30, 3), 'samples': sample})
    except OSError as e:
        report['checks']['candidate_cache_dirs'].append({'path': str(d), 'error': str(e)[:120]})
print('Cache candidates:', json.dumps(report['checks']['candidate_cache_dirs'], indent=2)[:6000])
print('Label-table candidates:', len(table_hits))


In [ ]:
# Report capacity and gates. A cache directory alone cannot establish training provenance.
raw_bytes = EXPECTED['studies_weak'] * EXPECTED['slices'] * EXPECTED['height'] * EXPECTED['width']
disk = shutil.disk_usage('/kaggle/working')
report['checks']['capacity'] = {'raw_weak_uint8_gib': round(raw_bytes / 2**30, 3), 'working_free_gib': round(disk.free / 2**30, 3)}
report['checks']['cuda_used'] = False
report['checks']['full_training_started'] = False
report['notes'].append('Input inventory is not proof that report labels, auxiliary models or ImageNet initialisation have competition-only provenance.')
report['notes'].append('The prior 05A 224-pixel cache does not match the 96-slice 384-pixel Raptor recipe.')
report['status'] = 'AUDIT_COMPLETE' if comp else 'NEEDS_COMPETITION'
report['next_gate'] = 'Review exact cache shape, label identity, provenance and GPU microbenchmark before any full training.'
report['elapsed_seconds'] = round(time.monotonic() - START, 2)
OUT.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding='utf-8')
print('Wrote', OUT, '| status', report['status'], '| seconds', report['elapsed_seconds'])
